# 📝 Graph·Vector 하이브리드 검색 과제 LV2(응용)

교안 02의 세 검색 경로 통합부터 PageRank와 인용 답변까지 직접 구현합니다.

- 1~3번: 줄거리 검색·GraphCypherQAChain·파라미터 조건 조회
- 4~5번: RRF 통합·필수 조건 적용·경로 비교
- 6~7번: PageRank 계산·같은 후보 재정렬
- 8~10번: 멀티홉 근거 구성·답변 생성·실제 인용 대조

<img src="images/04_lesson02_overview.png" width="1000">

**풀이 방법**: 교안 02 학습을 마친 뒤 새 커널에서 시작합니다. 각 문제의 빈 코드 셀에 함수·Cypher·실행 코드를 직접 작성하세요. 앞 문제의 결과는 다음 문제에서 이어 사용합니다. 힌트는 막혔을 때 펼치고, 각 주요 처리에는 짧은 주석을 붙이세요. 지문에 지정한 변수명·반환 키를 사용하고, 별도 지정이 없는 출력 형식은 자유입니다. 자가채점은 코드 결과를 확인하며, 검색 내용의 적합성은 원문을 읽고 판단합니다.

**데이터**: 기존 영화 그래프 171개 개체·253개 관계, 원문 38편·청크 571개를 사용합니다. 교안 01 본문에서 그래프와 두 검색 인덱스를 적재해 두세요. DB는 공유하며, 노트북 변수는 새로 만듭니다.

**모델 호출**: 정답 예시 기준으로 질문 임베딩·Jev 언어 판단·검색어 변환·Cypher 생성·답변 생성 각 1회입니다. 문서 벡터는 DB에 적재된 값을 사용합니다. 검색 순위와 모델 문장은 고정하지 않으며, 마지막에는 원문과 관계 근거를 직접 읽습니다.


## 준비와 데이터 연결

패키지와 `.env` 설정은 README를 따릅니다. 노트북 파일이 있는 폴더에서 시작하세요. 필요한 라이브러리를 가져옵니다.


In [ ]:
# 검색기와 결과 형식을 직접 작성할 라이브러리를 가져옵니다.
import json
import os
import re
from pathlib import Path
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from pydantic import BaseModel, Field
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem

from langchain_neo4j import GraphCypherQAChain, Neo4jGraph


이 폴더의 설정으로 DB·모델을 연결하고 원문을 읽습니다. `movies`는 전체 패킷, `movies["relations"]`는 관계 목록, `documents_by_id`는 원문 ID로 제목·본문·출처를 찾는 사전입니다.


In [ ]:
# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def run_cypher(query, **params):
    """파라미터 쿼리의 조회 결과를 사전 목록으로 반환합니다."""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


# 같은 자료의 원문과 관계 근거를 읽습니다.
movies = json.loads((material_dir / "data/movies_extraction_packet.json").read_text(encoding="utf-8"))
documents_by_id = movies["documents"]


## 1. 질문을 나누고 줄거리를 검색합니다

**배경**: 배우·연도 조건과 줄거리 조건을 각각 처리할 입력을 준비합니다.

**요구사항**:

- 전체 질문 `question`: “폴라 익스프레스(The Polar Express)와 배우를 1명 이상 공유하고 1999~2006년에 개봉한 영화 중, 루브르 박물관 살인사건의 암호와 종교적 비밀을 추적하는 기호학자의 이야기를 찾아 주세요. 공유 배우와 줄거리 근거도 알려 주세요.”
- 줄거리 부분만 담은 한글 `semantic_query`, 배우·연도 조건만 담은 한글 `graph_request`를 작성하세요.
- `filters`는 anchor(기준 영화의 영문 제목), min_year(시작 연도), max_year(끝 연도), min_shared(최소 공유 배우 수)를 키로 갖는 사전입니다. 연도와 배우 수는 정수로 담습니다.
- LV1에서 직접 작성한 `format_vector`, `vector_retriever`, `vector_documents`, `language_classifier`, `language_question`, `EnglishKeywords`, `lexical_prompt`, `lexical_chain`, `fulltext_documents`의 정의·생성 부분을 이 셀에 옮기세요. LV1의 질문과 검색 실행 코드는 가져오지 않습니다.
- semantic_query를 한 번 임베딩한 `query_vector`로 벡터 검색을 실행해 `dense_rows`를 만들고, 전문 검색 결과는 `fulltext_rows`에 담으세요. 각각 청크 20개를 조회합니다.

**확인 기준**: 줄거리 검색에 배우·연도 조건을 섞지 않고 두 검색 결과를 원문 ID별로 받습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 한 질문에서 본문 검색과 그래프 조건을 분리합니다.

세부구현:
1. 줄거리와 관계 조건을 각각 한국어 문장으로 작성합니다.
2. 직접 만든 두 검색 함수에 같은 줄거리 요구를 전달합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert filters == {"anchor": "The Polar Express", "min_year": 1999, "max_year": 2006, "min_shared": 1}
assert len(query_vector) == 768 and dense_rows and fulltext_rows
assert all(len(rows) == len({row["source_id"] for row in rows}) for rows in [dense_rows, fulltext_rows])
print("✅ 1번 핵심 확인 완료!")


## 2. GraphCypherQAChain으로 관계 후보를 찾습니다

**배경**: 자연어 관계 조건을 Cypher로 바꾸고 생성된 쿼리를 확인합니다.

**요구사항**:

- `neo4j_uri`와 환경변수 `NEO4J_USER`·`NEO4J_PASSWORD`로 `Neo4jGraph` 객체 `graph`를 만드세요. enhanced_schema는 False입니다.
- `cypher_prompt`를 직접 작성하세요. schema·question 입력을 받고 Movie·Person·Document·ACTED_IN·ABOUT_MOVIE만 사용하게 합니다. ACTED_IN은 인물에서 영화 방향이며, Document는 ABOUT_MOVIE로 영화에 연결됩니다. 기준 영화를 제외하고 연도·고유 배우 수 조건을 반영하며, 원문 id·title을 source_id·title로 반환하도록 지시하세요.
- `GraphCypherQAChain.from_llm`에 llm·graph·cypher_prompt를 연결해 `graph_chain`을 작성합니다. include_types는 위 다섯 유형, return_direct·return_intermediate_steps는 True, top_k는 100, allow_dangerous_requests는 True로 지정합니다. 이 옵션은 쓰기 권한을 제한하지 않습니다. 별도 실습 DB에서 실행하세요.
- 체인의 query 입력에 graph_request를 전달해 반환 사전을 `graph_response`에 담으세요. 이 사전의 result를 `graph_rows`, intermediate_steps 첫 항목의 query를 `generated_cypher`에 담고 쿼리와 결과를 출력합니다.

**확인 기준**: 생성된 Cypher가 관계 방향·기준 영화 제외·연도·고유 배우 수를 모두 반영해야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 프롬프트에 줄거리를 그래프 속성으로 추측하지 않도록 지시합니다.

세부구현:
1. 배우 수를 집계한 다음 최소 공유 배우 조건을 적용하게 합니다.
2. 생성 쿼리는 출력해 읽고 다음 문제의 파라미터 조회와 대조합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(graph_chain, GraphCypherQAChain)
assert generated_cypher.strip() and graph_rows
assert all({"source_id", "title"} <= row.keys() for row in graph_rows)
print("✅ 2번 핵심 확인 완료!")


## 3. 필수 조건과 두 출연 관계를 조회합니다

**배경**: 필수 조건에 맞는 후보와 배우별 양쪽 출연 근거를 확정합니다.

**요구사항**:

- `eligible_query`에 파라미터 Cypher를 직접 작성하세요. 파라미터 이름은 filters의 네 키와 같으며 연도는 양쪽 경계를 포함합니다. 기준 영화는 제외하며 고유 배우 수를 셉니다.
- 후보 원문 id인 source_id, 영화 title·released, standard_id인 movie_id, 고유 공유 배우 수인 shared_count, 출연 근거 paths를 반환합니다. paths의 각 사전에는 actor(배우 이름), first_claim(기준 영화 출연 claim_id), second_claim(후보 영화 출연 claim_id)을 담고 중복을 제거합니다. 후보 원문마다 한 행입니다.
- 조회 결과는 `eligible_rows`, 이 결과의 원문 ID 집합은 `allowed_ids`로 만드세요.
- 생성 쿼리에만 있는 ID는 `generated_only`, 조건 조회에만 있는 ID는 `condition_only`로 구해 출력합니다. 차이가 있으면 생성 쿼리의 조건을 읽고 프롬프트를 개선하세요.

**확인 기준**: 연도·배우 수 조건과 두 출연 관계가 맞아야 하며, 두 조회의 차이가 있으면 원인을 설명합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 기준 영화와 후보 영화 사이를 인물에서 나가는 두 ACTED_IN 관계로 연결합니다.

세부구현:
1. 후보별 고유 배우 수와 관계 근거를 모읍니다.
2. 후보 개체를 원문으로 연결한 뒤 같은 원문 ID 집합으로 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert eligible_rows and allowed_ids == {row["source_id"] for row in eligible_rows}
assert len(eligible_rows) == len(allowed_ids)
assert all(filters["min_year"] <= row["released"] <= filters["max_year"] and row["title"] != filters["anchor"] and row["shared_count"] >= filters["min_shared"] for row in eligible_rows)
assert all(row["shared_count"] == len({path["actor"] for path in row["paths"]}) for row in eligible_rows)
assert all(
    any(relation["claim_id"] == path[key] and relation["relation"] == "ACTED_IN"
        and relation["source_record"]["source"] == path["actor"]
        and relation["source_record"]["target"] == title for relation in movies["relations"])
    for row in eligible_rows for path in row["paths"]
    for key, title in [("first_claim", filters["anchor"]), ("second_claim", row["title"])])
assert generated_only == {row["source_id"] for row in graph_rows} - allowed_ids
assert condition_only == allowed_ids - {row["source_id"] for row in graph_rows}
print("✅ 3번 핵심 확인 완료!")


## 4. 세 검색 경로를 RRF로 합칩니다

**배경**: 서로 다른 점수 체계의 검색 결과를 같은 원문 ID로 통합합니다.

**요구사항**:

- `fuse_routes(routes, weights, c=60)`를 작성하세요. 입력 routes는 경로명을 키로, 검색 행 목록을 값으로 갖는 사전입니다. 각 경로 안에서 source_id가 처음 나온 순서를 유지하며 중복을 제거한 뒤 순위를 1부터 셉니다.
- 전문·벡터는 해당 순위, 관계 경로는 모든 후보에 순위 1을 적용합니다. 각 경로가 더할 점수는 가중치 / (c + 순위)입니다.
- 출력은 모든 경로의 원문을 source_id별로 합친 사전 목록입니다. 각 행에는 source_id, 경로별 점수 fulltext·dense·graph, 그 세 점수의 합인 rrf_score를 담으세요. 없는 경로 점수는 0입니다. 총점 내림차순, 동점이면 source_id순으로 반환합니다.
- `routes`의 fulltext·dense·graph 키에 각각 fulltext_rows·dense_rows·graph_rows를 담고 `weights`는 fulltext 0.3, dense 0.5, graph 0.2로 정합니다. 통합 결과를 `fused_rows`에 담으세요.

**확인 기준**: 같은 경로의 중복 문서는 점수가 늘지 않고, 관계 후보에는 동일한 점수를 더합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 원래 점수를 더하지 않고 경로별 문서 순위를 사용합니다.

세부구현:
1. 중복 기록은 경로가 바뀔 때마다 새로 시작합니다.
2. 경로별 점수와 합계를 함께 기록해 통합 결과를 설명합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert len(fused_rows) == len({row["source_id"] for rows in routes.values() for row in rows})
assert all(abs(next(row[name] for row in fused_rows if row["source_id"] == source_id) - weights[name] / (60 + rank)) < 1e-12 for name in ["fulltext", "dense"] for rank, source_id in enumerate(dict.fromkeys(item["source_id"] for item in routes[name]), 1))
assert all(row[name] == 0 for row in fused_rows for name in routes if row["source_id"] not in {item["source_id"] for item in routes[name]})
assert all(abs(row["rrf_score"] - row["fulltext"] - row["dense"] - row["graph"]) < 1e-12 for row in fused_rows)
assert all(abs(row["graph"] - weights["graph"] / 61) < 1e-12 for row in fused_rows if row["source_id"] in {item["source_id"] for item in graph_rows})
assert fuse_routes({name: rows + rows for name, rows in routes.items()}, weights) == fused_rows
assert fuse_routes({**routes, "graph": list(reversed(graph_rows))}, weights) == fused_rows
assert all(fused_rows[i]["rrf_score"] >= fused_rows[i + 1]["rrf_score"] for i in range(len(fused_rows) - 1))
print("✅ 4번 핵심 확인 완료!")


## 5. 필수 조건을 적용하고 경로 조합을 비교합니다

**배경**: 가중치와 무관하게 배우·연도 조건을 지키는 후보만 남깁니다.

**요구사항**:

- fused_rows 중 allowed_ids에 속한 행만 순서대로 남겨 `candidates`에 담으세요.
- 같은 검색 결과를 재사용해 전문·벡터 두 경로만 합친 `text_fused`를 만드세요. 두 경로의 원래 가중치 비율을 유지하면서 합이 1이 되도록 다시 계산합니다. 조건을 통과한 목록은 `text_candidates`입니다.
- 세 경로 후보에만 있는 원문 ID 집합을 `graph_added_ids`에 담으세요. 두 조합의 후보 제목과 추가된 ID를 출력합니다.

**확인 기준**: 두 조합 모두 동일한 필수 조건을 만족하고 검색을 다시 호출하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 검색의 보너스와 필수 조건 필터는 역할이 다릅니다.

세부구현:
1. 기존 routes에서 비교할 경로를 선택하고 비중의 합을 맞춥니다.
2. 같은 allowed_ids로 제한한 뒤 후보 ID 집합을 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert text_fused == fuse_routes(
    {name: routes[name] for name in ["fulltext", "dense"]},
    {name: weights[name] / (weights["fulltext"] + weights["dense"]) for name in ["fulltext", "dense"]})
assert candidates == [row for row in fused_rows if row["source_id"] in allowed_ids]
assert text_candidates == [row for row in text_fused if row["source_id"] in allowed_ids]
assert graph_added_ids == {row["source_id"] for row in candidates} - {row["source_id"] for row in text_candidates}
assert all(row["source_id"] in allowed_ids for row in candidates + text_candidates)
print("✅ 5번 핵심 확인 완료!")


## 6. 출연 그래프의 PageRank를 계산합니다

**배경**: 영화가 출연 관계 그래프에서 얼마나 중심적인지 구합니다.

**요구사항**:

- `projection_name`은 `assignment_movie_cast`로 지정합니다. `gds.graph.drop`의 두 번째 인자를 False로 두어 같은 이름의 메모리 그래프를 제거한 뒤 새로 만드세요.
- Movie·Person과 ACTED_IN을 GDS 메모리 그래프에 담습니다. 계산할 때 관계는 UNDIRECTED로 처리합니다.
- dampingFactor 0.85로 PageRank를 계산해 `pagerank` 속성에 저장하세요.
- 문서가 설명하는 **영화 자체**의 점수를 조회해 source_id·title·pagerank가 있는 `pagerank_rows`를 만드세요. source_id로 점수를 찾는 `pagerank_by_id` 사전을 작성합니다.

**확인 기준**: 38개 원문마다 해당 영화의 양수 PageRank 점수가 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 원문·청크 노드는 이번 PageRank 계산에 넣지 않습니다.

세부구현:
1. 영화·인물 사이 출연 관계를 양방향으로 계산합니다.
2. 조회할 때 원문에서 ABOUT_MOVIE로 영화의 점수를 가져옵니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert len(pagerank_rows) == 38 and len(pagerank_by_id) == 38
assert set(pagerank_by_id) == set(documents_by_id)
assert all(value > 0 for value in pagerank_by_id.values())
assert all(pagerank_by_id[row["source_id"]] == row["pagerank"] for row in pagerank_rows)
print("✅ 6번 핵심 확인 완료!")


## 7. 같은 후보에서 PageRank 비중을 비교합니다

**배경**: 후보 집합을 고정한 상태에서 검색 관련성과 그래프 중요도를 결합합니다.

**요구사항**:

- `minmax(values)`는 비어 있지 않은 숫자 목록을 받아 (현재 값 - 최솟값) / (최댓값 - 최솟값)을 원래 순서의 목록으로 반환합니다. 모든 값이 같으면 0.5씩 반환하세요.
- `rerank_candidates(rows, weight)`를 작성하세요. rows는 5번 candidates와 같은 형식입니다. 후보의 rrf_score와 pagerank_by_id에서 source_id로 찾은 영화 PageRank를 각각 현재 후보 안에서 정규화해 rrf_norm·pagerank_norm에 담습니다. final_score는 (1-weight) × rrf_norm + weight × pagerank_norm입니다.
- 입력 행의 기존 키를 유지한 새 사전에 세 점수를 추가하며, final_score 내림차순·동점이면 source_id순으로 반환합니다. 빈 후보는 빈 목록을 반환합니다.
- 같은 candidates에 비중 0·0.25·0.65를 적용한 결과를 `rankings` 사전에 담으세요. 키는 해당 숫자입니다. 비중별 제목과 final_score를 출력합니다. 각 비중의 첫 영화 원문은 Neo4j에서 조회해 읽으세요. 같은 영화가 반복되면 원문은 한 번만 출력합니다.

**확인 기준**: 세 비중에서 후보 ID는 같고, 비중 0에서는 RRF 정규화 점수가 최종 점수입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 종류 점수를 같은 후보 안에서 각각 정규화합니다.

세부구현:
1. 원본 후보를 수정하지 않도록 새 사전을 만듭니다.
2. 후보 원문을 읽어 순위 변화가 줄거리 관련성을 높였는지 판단합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert minmax([2, 2]) == [0.5, 0.5] and minmax([1, 3]) == [0.0, 1.0]
assert minmax([2, 4, 10]) == [0.0, 0.25, 1.0]
assert all(rows == sorted(rows, key=lambda row: (-row["final_score"], row["source_id"])) for rows in rankings.values())
assert all(row["pagerank_norm"] == minmax([pagerank_by_id[candidate["source_id"]] for candidate in candidates])[[candidate["source_id"] for candidate in candidates].index(row["source_id"])] for rows in rankings.values() for row in rows)
assert rerank_candidates([], 0.25) == []
assert set(rankings) == {0, 0.25, 0.65}
assert all({row["source_id"] for row in rows} == {row["source_id"] for row in candidates} for rows in rankings.values())
assert all(0 <= row["rrf_norm"] <= 1 and 0 <= row["pagerank_norm"] <= 1 and abs(row["final_score"] - (1 - weight) * row["rrf_norm"] - weight * row["pagerank_norm"]) < 1e-12 for weight, rows in rankings.items() for row in rows)
assert all(row["rrf_norm"] == minmax([candidate["rrf_score"] for candidate in candidates])[[candidate["source_id"] for candidate in candidates].index(row["source_id"])] for rows in rankings.values() for row in rows)
assert all("final_score" not in row for row in candidates)
print("✅ 7번 핵심 확인 완료!")


**판단**: 세 비중의 첫 영화 원문을 읽고, 이 질문에 사용할 비중을 이유와 함께 적으세요. 첫 영화가 같다면 점수 차이를 비교하세요.


*(여기에 판단과 근거를 서술하세요)*


## 8. 멀티홉 결과를 원문과 관계 근거로 재구성합니다

**배경**: 줄거리와 연도, 공유 배우라는 서로 다른 주장을 증명할 자료를 모읍니다.

**요구사항**:

- DB에 적재한 동일 배포 패킷의 `movies["relations"]`에서 claim_id를 키, 관계 레코드 전체를 값으로 갖는 `relations_by_id` 사전을 만드세요.
- `build_evidence(ranked, eligible_rows, top_n=3)`를 작성하세요. ranked의 앞 top_n개를 source_id로 documents_by_id·eligible_rows와 연결하세요. 원문과 영화 속성을 담고, 해당 후보 paths의 first_claim·second_claim에 해당하는 관계 근거를 추가한 사전 목록을 반환합니다.
- 모든 항목의 공통 키는 evidence_id·kind·url·text입니다. 원문은 kind=plot, ID=source_id, text=원문 전체, url=원문의 url이며 title도 담습니다. 영화 속성은 kind=movie_record, ID는 movie: 뒤에 movie_id, text에는 제목·개봉연도를 담습니다. URL은 `https://github.com/neo4j-graph-examples/movies`입니다.
- 출연 관계는 kind=cast_relation, ID=claim_id, text와 url에는 관계 레코드의 evidence와 source_url 값을 담습니다. 여러 후보가 같은 관계를 사용해도 전체 목록에 한 번만 넣습니다.
- 비중 0.25의 rankings 결과를 `ranked`에 담고 상위 3편의 `evidence`를 만드세요. ID·종류·출처를 출력합니다.

**확인 기준**: 상위 후보의 줄거리·연도·양쪽 출연 관계가 모두 실제 데이터와 연결되고 ID가 중복되지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 경로의 ID를 원본 관계 근거와 연결합니다.

세부구현:
1. 원문과 영화 속성을 먼저 담고 필요한 관계 ID를 집합에 모읍니다.
2. 모은 관계 ID로 근거 문장과 출처를 가져옵니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert evidence and len(evidence) == len({item["evidence_id"] for item in evidence})
assert {item["evidence_id"] for item in evidence if item["kind"] == "movie_record"} == {"movie:" + row["movie_id"] for row in eligible_rows if row["source_id"] in {candidate["source_id"] for candidate in ranked[:3]}}
assert all(row["title"] in item["text"] and str(row["released"]) in item["text"] for row in eligible_rows for item in evidence if item["evidence_id"] == "movie:" + row["movie_id"])
assert {item["evidence_id"] for item in evidence if item["kind"] == "plot"} == {row["source_id"] for row in ranked[:3]}
assert all({"evidence_id", "kind", "url", "text"} <= item.keys() and item["url"] and item["text"] for item in evidence)
assert {item["evidence_id"] for item in evidence if item["kind"] == "cast_relation"} == {path[key] for row in eligible_rows if row["source_id"] in {candidate["source_id"] for candidate in ranked[:3]} for path in row["paths"] for key in ["first_claim", "second_claim"]}
assert all(item["text"] == relations_by_id[item["evidence_id"]]["evidence"] for item in evidence if item["kind"] == "cast_relation")
assert all(item["text"] == documents_by_id[item["evidence_id"]]["text"] for item in evidence if item["kind"] == "plot")
print("✅ 8번 핵심 확인 완료!")


## 9. 근거를 인용하는 답변 체인을 작성합니다

**배경**: 모델이 실제 검색한 자료로만 조건에 맞는 영화를 설명하도록 연결합니다.

**요구사항**:

- `GroundedAnswer`는 answer 문자열과 evidence_ids 문자열 목록만 갖는 BaseModel로 작성하세요.
- `answer_prompt`를 직접 작성합니다. question·context 입력을 사용하고 한국어로 작품·연도, 공유 배우, 줄거리의 세 항목을 답하게 하세요. 본문 문장 뒤에 실제 ID마다 별도 대괄호로 인용하고 evidence_ids에는 사용한 모든 ID를 빠짐없이 한 번씩 담게 합니다.
- 연도는 movie_record, 줄거리는 plot, 공유 배우는 양쪽 cast_relation을 인용하도록 지시하세요. 배우와 극중 인물을 구분하고, 근거가 부족한 내용은 확인할 수 없다고 답하게 합니다.
- 프롬프트와 llm의 구조화 출력을 연결한 `answer_chain`을 만드세요. question 입력에는 1번의 전체 질문 문자열을 그대로, context 입력에는 evidence만 JSON 문자열로 바꿔 전달합니다. 반환된 GroundedAnswer 객체를 `answer`에 담고 answer 속성의 본문을 출력하세요.

**확인 기준**: 한글 답변에 실제 근거 ID가 표시되고, 인용 목록에 제공하지 않은 ID가 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 답변 구조와 인용 규칙을 프롬프트에 함께 담습니다.

세부구현:
1. 구조화 출력은 본문과 근거 ID 목록 두 필드만 사용합니다.
2. 후보 제목만 전달하지 말고 실제 evidence 전체를 전달합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(answer, GroundedAnswer) and answer.answer.strip()
assert re.search(r"[가-힣]", answer.answer)
assert answer.evidence_ids and set(answer.evidence_ids) <= {item["evidence_id"] for item in evidence}
assert all(f"[{evidence_id}]" in answer.answer for evidence_id in answer.evidence_ids)
print("✅ 9번 핵심 확인 완료!")


## 10. 인용 원문과 답변의 주장을 대조합니다

**배경**: ID가 존재하는지와 그 근거가 실제 주장을 뒷받침하는지를 구분합니다.

**요구사항**:

- `evidence`의 evidence_id를 키, 해당 근거 사전을 값으로 갖는 `evidence_by_id`를 만드세요.
- `answer.answer`의 대괄호 안 ID를 정규표현식으로 추출해 집합 `inline_ids`에 담으세요. `unknown_ids`는 본문과 answer.evidence_ids 양쪽의 ID 중 evidence_by_id에 없는 ID의 집합입니다. `citation_gap`은 본문 ID와 모델 목록 중 한쪽에만 있는 ID의 집합입니다.
- 모델 목록의 ID 중 evidence_by_id에 있는 근거의 종류·출처·본문을 출력하세요. 없는 ID는 unknown_ids로 확인합니다. ID만 맞고 내용은 다를 수 있으므로 아래 판단도 작성합니다.

**확인 기준**: unknown_ids·citation_gap이 비어 있고, 줄거리·연도·공유 배우에 해당하는 근거를 직접 대조합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 존재하는 ID를 인용해도 문장이 근거에 없는 주장을 할 수 있습니다.

세부구현:
1. 본문에서 추출한 ID 집합과 모델의 목록을 비교합니다.
2. 원문에서 연도와 양쪽 출연 관계, 암호를 추적하는 과정을 읽습니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert set(evidence_by_id) == {item["evidence_id"] for item in evidence}
assert inline_ids == set(re.findall(r"\[([^\[\]]+)\]", answer.answer))
assert unknown_ids == (inline_ids | set(answer.evidence_ids)) - set(evidence_by_id)
assert citation_gap == inline_ids ^ set(answer.evidence_ids)
assert not unknown_ids and not citation_gap
assert {evidence_by_id[evidence_id]["kind"] for evidence_id in answer.evidence_ids} == {"plot", "movie_record", "cast_relation"}
print("✅ 10번 핵심 확인 완료!")


**판단**: 작품·연도, 공유 배우의 양쪽 출연 관계, 암호를 추적하는 과정을 근거 ID와 함께 3문장으로 적으세요. 또한 후보 단계에 정답이 없을 때 PageRank 비중만 바꾸면 해결되는지 설명하세요.


*(여기에 판단과 근거를 서술하세요)*
